# Read LSSTCamSources in all bands

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-09
- **Last update:** 2026-07-11 : add statistic plots like box and violin

## Goal

This notebook is the **offline companion** to `01_FindLSSTCamSourcesInAllbands.ipynb`.
It does **not** touch the Butler: it simply reads back the per-band parquet
files (`objectstats_band_<band>.parquet`) written by notebook 01 into
`OUTPUT_DIR`, reproduces the same summary plots, and adds a new diagnostic
figure: a 2x3 grid of 2D histograms (dispersion `mmag_meas` vs. mean
magnitude) for each band, in the standard LSST order `u, g, r, i, z, y`,
designed to make the high-dispersion tail of the distribution visible in
each band.

**Note:** the parquet files produced at USDF are not copied into this local
directory; point `OUTPUT_DIR` below to wherever you have synced/copied the
`output_objectstats/` folder (or run this notebook directly at USDF).

## 1. Imports

In [ ]:
import glob
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit

In [ ]:
# vieww all contents of pandas tables
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
# try:
#    import ipympl  # noqa: F401
#
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → %matplotlib inline")

In [ ]:
# ipympl not working
# %matplotlib inline

In [ ]:
import ipywidgets

%matplotlib widget

import ipywidgets as widgets

widgets.IntSlider()

print("matplotlib:", matplotlib.__version__)
print("backend:", matplotlib.get_backend())
print("ipywidgets:", ipywidgets.__version__)

In [ ]:
# test
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [1, 4, 9])
plt.show()

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Le logging est configure et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the directory containing the
`objectstats_band_<band>.parquet` files written by notebook 01
(`OUTPUT_DIR` there). These constants are only used for plot titles / labels
here (the actual selection was already applied in notebook 01).

In [ ]:
# -- Where notebook 01 wrote its output --------------------------------------


# ── Notebook tag ──────────────────────────────────────────────────────
NB_TAG = "PlotLSSTCamSources_02"

# ── Input: merged LC files from notebook 01 ───────────────────────────────
DIR_DATA_IN = "./data_FindLSSTCamSources_01"


# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)


# All LSST bands, in the standard display order
BANDS = ["u", "g", "r", "i", "z", "y"]
BANDS_CMAP = {"u": "Purples", "g": "Greens", "r": "Reds", "i": "YlOrBr", "z": "pink_r", "y": "bone_r"}
BANDS_COLOR = {
    "u": "blueviolet",
    "g": "limegreen",
    "r": "red",
    "i": "darkorange",
    "z": "chocolate",
    "y": "saddlebrown",
}

# -- Magnitude window used in notebook 01 (for plot titles only) -------------
MAG_MIN = 17.0
MAG_MAX = 19.5

# -- Minimum number of visits per band used in notebook 01 (for plot titles) -
MIN_VISITS_PER_BAND = {"u": 20, "g": 50, "r": 50, "i": 50, "z": 50, "y": 20}

log.info(f"Reading per-band object-stats parquet files from '{DIR_DATA_IN}'")

## 4. Helper functions

In [ ]:
# ── savefig: PDF + PNG ───────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
# helpers for statistics


def sigma_iqr(x):
    """Robust scatter estimator: interquartile range rescaled so that it
    matches the standard deviation for a pure Gaussian distribution."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    q25, q75 = np.percentile(x, [25, 75])
    return (q75 - q25) / 1.349

In [ ]:
# helpers for fitting
def gaussian(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def fit_gaussian_to_hist(data, n_bins=40, x_range=None):
    """Fit a Gaussian to the histogram of `data`.

    Returns a dict with keys `mu`, `sigma`, `amp`, `edges` on success, or
    None if the fit could not be performed (too few points or curve_fit
    failure).
    """
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if len(data) < 10:
        return None

    if x_range is None:
        x_range = (0.0, np.nanpercentile(data, 99.0))

    counts, edges = np.histogram(data, bins=n_bins, range=x_range)
    centers = 0.5 * (edges[:-1] + edges[1:])

    siqr = sigma_iqr(data)
    p0 = [max(counts.max(), 1.0), np.median(data), siqr if siqr > 0 else np.std(data)]

    try:
        popt, _ = curve_fit(gaussian, centers, counts, p0=p0, maxfev=5000)
    except Exception:
        return None

    amp, mu, sigma = popt
    return {"amp": amp, "mu": mu, "sigma": abs(sigma), "edges": edges}

## 5. Read the per-band parquet files

Each file was written by notebook 01 as
`OUTPUT_DIR/objectstats_band_<band>.parquet` and already contains one row per
stable star (object) with the columns produced by `summarize_objects()`:
`object_id, n_visits, ra, dec, flux_mean, flux_std, mag_mean,
sigmaF_over_F_phot, sigmaF_over_F_meas, mmag_meas, mmag_phot, ddf, band`.

In [ ]:
all_band_results = {}

for band in BANDS:
    path = os.path.join(DIR_DATA_IN, f"objectstats_band_{band}.parquet")
    if not os.path.exists(path):
        log.warning(f"Band '{band}': file not found ({path}), skipping")
        continue
    df_band = pd.read_parquet(path)
    all_band_results[band] = df_band
    log.info(f"Band '{band}': {len(df_band)} objects read from {path}")

if not all_band_results:
    raise FileNotFoundError(
        f"No objectstats_band_*.parquet files found in '{DIR_DATA_IN}'. "
        "Check DIR_DATA_IN  in the configuration cell above."
    )

## 6. Combine all bands and inspect the results

In [ ]:
df_all = pd.concat(
    [df for df in all_band_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total objects across all bands/DDFs: {len(df_all)}")

df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

## 7 Plots

### 7.1  Plot: relative photometric scatter (mmag) per band

Boxplot of `mmag_meas` (measured scatter, using `psfFlux`) grouped by band,
in the standard LSST band order `u, g, r, i, z, y`. We expect the largest
scatter in **u** and **y**.

In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
flier_props_71 = dict(
    marker="o",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="lightgray",
    markeredgewidth=1.0,
    alpha=0.6,
)
ax.boxplot(data, tick_labels=band_order, showfliers=True, flierprops=flier_props_71)

ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.0, 200.0)

plt.tight_layout()

savefig(fig, "mmag_scatter_allsrc_perband")

plt.show()

In [ ]:
# Cross-check: measured scatter vs. photon-noise-only expectation, per band
fig, ax = plt.subplots(figsize=(8, 6))
for b in band_order:
    sub = df_all.loc[df_all["band"] == b]
    ax.scatter(sub["mag_median"], sub["mmag_meas"], s=10, color=BANDS_COLOR[b], alpha=0.4, label=b)
ax.set_xlabel("mean magnitude")
ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")
ax.set_yscale("log")
ax.legend(markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()

savefig(fig, "mmag_scatter_allsrc_allbands")

plt.show()

### 7.2  Dispersion vs. magnitude, 2D histogram per band

New figure requested: a `2 x 3` grid of 2D histograms of `mmag_meas`
(dispersion, log-binned) vs. `mag_mean` (magnitude, linear-binned), one panel
per band, in the order `u, g, r, i, z, y`. The dispersion axis is log-scaled
(both the bin edges and the axis) so that the high-dispersion tail -- objects
whose repeated-visit flux scatter is much larger than the bulk of the
population, e.g. blends, variables, or bad cross-matches -- stands out
clearly in every band, rather than being compressed against the x-axis as it
would be on a linear scale.

In [ ]:
def plot_disp_vs_mag(ax, mag, disp, band, mag_min=MAG_MIN, mag_max=MAG_MAX, n_xbins=50, n_ybins=50):
    """2D histogram of dispersion (mmag_meas, log-binned) vs magnitude
    (linear-binned) on a single axis, with a log-scaled y-axis so the
    high-dispersion tail is visible.
    """
    mag = np.asarray(mag, dtype=float)
    disp = np.asarray(disp, dtype=float)
    sel = np.isfinite(mag) & np.isfinite(disp) & (disp > 0)
    mag, disp = mag[sel], disp[sel]

    if len(disp) == 0:
        ax.set_title(f"band {band} (no data)")
        return None

    xedges = np.linspace(mag_min, mag_max, n_xbins + 1)

    # log-spaced bins on the dispersion axis, padded slightly beyond the
    # 0.5th/99.5th percentiles so the tail is not clipped at the bin edge
    ylo = max(np.nanpercentile(disp, 0.5) * 0.8, disp[disp > 0].min())
    yhi = np.nanpercentile(disp, 99.5) * 1.5
    yedges = np.logspace(np.log10(ylo), np.log10(yhi), n_ybins + 1)

    h, xe, ye = np.histogram2d(mag, disp, bins=[xedges, yedges])
    mesh = ax.pcolormesh(xe, ye, h.T, norm=LogNorm(vmin=1, vmax=max(h.max(), 1)), cmap=BANDS_CMAP[band])
    ax.set_yscale("log")
    ax.set_xlim(mag_min, mag_max)
    ax.set_title(f"band {band}  (N={len(disp)})")
    ax.grid(True, alpha=0.2, which="both")
    return mesh


fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)

band_grid_order = ["u", "g", "r", "i", "z", "y"]

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue
    sub = df_all.loc[df_all["band"] == band]
    mesh = plot_disp_vs_mag(ax, sub["mag_median"], sub["mmag_meas"], band)
    if mesh is not None:
        fig.colorbar(mesh, ax=ax, label="N objects")

for ax in axes[1, :]:
    ax.set_xlabel("mean magnitude")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")

fig.suptitle(
    f"Photometric scatter vs. magnitude per band, {MAG_MIN:.1f} < mag < {MAG_MAX:.1f}",
    y=1.0,
    fontsize=20,
)
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, "mmag_vs_mag_hist2d_per_band.png"), dpi=150, bbox_inches="tight")
# plt.savefig(os.path.join(OUTPUT_DIR, "mmag_vs_mag_hist2d_per_band.pdf"), bbox_inches="tight")

savefig(fig, "mmag_vs_mag_hist2d_per_band")

plt.show()

### 7.3  Dispersion distribution per magnitude bin, one figure per band

For each band, split the magnitude range `[MAG_MIN, MAG_MAX]` into bins of
width `MAG_BIN_WIDTH` (0.5 mag by default) and, for each bin, plot the
histogram of the measured dispersion `mmag_meas`. Subplots are stacked
**vertically** (one row per magnitude bin, faintest at the bottom) so the
broadening of the distribution with increasing magnitude is directly
visible from top to bottom.

For each magnitude bin we compute:
- `sigma_IQR = (Q75 - Q25) / 1.349`, a robust scatter estimator insensitive
  to the high-dispersion tail (interquartile range rescaled to match the
  standard deviation for a pure Gaussian).
- A **Gaussian fit** to the histogram (`scipy.optimize.curve_fit`), restricted
  to `[0, 99th percentile]` of the bin so the long tail does not dominate the
  fit; the fitted `sigma_fit` is reported alongside `sigma_IQR` for
  comparison -- a much larger `sigma_fit` than `sigma_IQR` would flag a
  non-Gaussian tail rather than a genuinely broader core.

#### Do the plot of histograms in magnitude slices

In [ ]:
# deine the size of the slices
MAG_BIN_WIDTH = 0.5  # mag


mag_edges = np.arange(MAG_MIN, MAG_MAX + 1e-9, MAG_BIN_WIDTH)
if mag_edges[-1] < MAG_MAX - 1e-9:
    mag_edges = np.append(mag_edges, MAG_MAX)
n_mag_bins = len(mag_edges) - 1

log.info(f"{n_mag_bins} magnitude bins of width {MAG_BIN_WIDTH} mag: {np.round(mag_edges, 2).tolist()}")


# loop on band, one figure per band
for band in band_grid_order:
    if band not in df_all["band"].unique():
        log.warning(f"Band '{band}': no data, skipping figure")
        continue

    # select the band
    sub_band = df_all.loc[df_all["band"] == band]

    # create one figure per band
    fig, axes = plt.subplots(
        n_mag_bins,
        1,
        figsize=(8, 2 * n_mag_bins),
        sharex=True,
    )

    if n_mag_bins == 1:
        axes = [axes]

    # loop on magnitude bands
    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        mag_center = np.mean([lo, hi])
        mag_halfwidth = (hi - lo) / 2.0

        ax = axes[k]

        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]

        if len(disp) < 5:
            ax.set_title(
                f"{lo:.1f} <= mag < {hi:.1f}  (N={len(disp)}, too few objects)", fontsize=9, loc="left"
            )
            ax.set_ylabel("N")
            continue

        median = np.median(disp)
        siqr = sigma_iqr(disp)
        x_range = (0.0, np.nanpercentile(disp, 99.0))

        ax.hist(
            disp,
            bins=40,
            range=x_range,
            color=BANDS_COLOR[band],
            alpha=0.75,
            edgecolor="white",
            linewidth=0.3,
        )

        label = f"{lo:.1f} <= mag < {hi:.1f}  (N={len(disp)})   " + rf"$\sigma_{{IQR}}$={siqr:.1f} mmag"

        # add a textbox for better readability
        textstr = "\n".join(
            (
                f"band : {band},",
                f" mag-bin : mag = {mag_center} +/- {mag_halfwidth} mag",
                f" dispersion : median = {median:.1f} mmag",
                f" dispersion : sigma_iqr = {siqr:.1f} mmag",
            )
        )

        props = dict(boxstyle="round", alpha=0.5, facecolor="white")

        # place a text box in upper left in axes coords
        ax.text(0.55, 0.9, textstr, transform=ax.transAxes, fontsize=10, verticalalignment="top", bbox=props)

        fit = fit_gaussian_to_hist(disp, n_bins=40, x_range=x_range)
        if fit is not None:
            xx = np.linspace(x_range[0], x_range[1], 300)
            ax.plot(
                xx,
                gaussian(xx, fit["amp"], fit["mu"], fit["sigma"]),
                color="crimson",
                lw=2,
                label="Gaussian fit",
            )
            label += rf"   $\sigma_{{fit}}$={fit['sigma']:.1f} mmag"
            ax.legend(loc="upper right", fontsize=8, frameon=False)

        ax.set_title(label, fontsize=9, loc="left")
        ax.set_ylabel("N")
        ax.set_xlim(*x_range)
        ax.grid(True, alpha=0.2)

    axes[-1].set_xlabel(r"$\sigma_F/F$ (mmag), measured")
    fig.suptitle(
        f"Band '{band}': dispersion distribution per {MAG_BIN_WIDTH:.1f}-mag bin "
        "(faintest bin at the bottom)",
        y=1.00,
        fontsize=12,
    )
    plt.tight_layout()
    # plt.savefig(
    #    os.path.join(OUTPUT_DIR, f"mmag_hist_by_magbin_band_{band}.png"), dpi=150, bbox_inches="tight"
    # )
    # plt.savefig(os.path.join(OUTPUT_DIR, f"mmag_hist_by_magbin_band_{band}.pdf"), bbox_inches="tight")

    savefig(fig, f"mmag_hist_by_magbin_band_{band}.pdf")

    plt.show()

### 7.4  Median dispersion vs. magnitude, error-bar plot (all bands, single axis)

Single-axis (no subplot) error-bar figure: for each band and each
magnitude bin (`mag_edges`, same binning as section 7.3), we plot one point
at `(mag_center, median(mmag_meas))`, with:
- **x-error** = half the magnitude-bin width (`mag_halfwidth`),
- **y-error** = the robust scatter `sigma_IQR` of `mmag_meas` in that bin.

All six bands are overlaid on the same axis, using the same `BANDS_COLOR`
color code as the rest of the notebook.

In [ ]:
# ── 7.4  Median dispersion vs. magnitude, per band, single axis ────────
# One point per (band, magnitude bin): x = mag_center +/- mag_halfwidth,
# y = median(mmag_meas) +/- sigma_IQR(mmag_meas), using the same mag_edges
# as section 7.3.

fig, ax = plt.subplots(figsize=(8, 6))

for band in band_grid_order:
    if band not in df_all["band"].unique():
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    mag_centers, mag_halfwidths, medians, siqrs = [], [], [], []

    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]

        if len(disp) < 5:
            continue

        mag_centers.append(np.mean([lo, hi]))
        mag_halfwidths.append((hi - lo) / 2.0)
        medians.append(np.median(disp))
        siqrs.append(sigma_iqr(disp))

    if len(mag_centers) == 0:
        continue

    ax.errorbar(
        mag_centers,
        medians,
        xerr=mag_halfwidths,
        yerr=siqrs,
        fmt="o",
        color=BANDS_COLOR[band],
        ecolor=BANDS_COLOR[band],
        elinewidth=1.2,
        capsize=3,
        markersize=10,
        label=band,
    )

ax.set_xlabel("mean magnitude")
ax.set_ylabel(r"median $\sigma_F/F$ (mmag), measured")
ax.set_title(
    f"Median photometric scatter vs. magnitude per band "
    f"({MAG_BIN_WIDTH:.1f}-mag bins, error bars = $\\sigma_{{IQR}}$)"
)
ax.legend(title="band", ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()

savefig(fig, "mmag_median_vs_mag_errorbar_allbands")

plt.show()

### 7.5  Dispersion vs. magnitude bin, boxplots per band (2x3 grid)

`2 x 3` grid of boxplots, one panel per band (order `u, g, r, i, z, y`),
showing the distribution of `mmag_meas` in each magnitude bin
(`mag_edges`, same binning as sections 7.3/7.4). Boxplots make outliers
explicit (individual points beyond `1.5x` the interquartile range), which is
the goal here: transient atmospheric-transmission variations (clouds, poor
photometric conditions) should show up as an excess of high-dispersion
outlier points in a given magnitude bin. Each panel is colored using the
notebook's `BANDS_COLOR` code. We expect more/larger outliers in **u** and
**y**, somewhat fewer in **g** and **z**, and very few in **r** and **i**.

In [ ]:
# ── 7.5  Dispersion vs. magnitude bin, boxplots per band ───────────────
# One boxplot panel per band, x-axis = magnitude bin, y-axis = mmag_meas,
# outliers shown explicitly (matplotlib default: points beyond 1.5x IQR),
# drawn as large open circles so they stand out clearly.

# -- Optional parameter: common y-axis maximum (mmag) shared by all 6 panels,
# so the bands can be visually compared on the same scale. Set to a number
# (e.g. 60.0) to fix it manually, or leave as None to auto-scale it from the
# data (99.5th percentile across all bands, with a small margin).
BOXPLOT_YMAX = 200  # e.g. 60.0

mag_bin_labels = [f"{mag_edges[k]:.1f}-{mag_edges[k + 1]:.1f}" for k in range(n_mag_bins)]

if BOXPLOT_YMAX is None:
    ymax_common = np.nanpercentile(df_all["mmag_meas"].dropna(), 99.5) * 1.1
else:
    ymax_common = BOXPLOT_YMAX

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

median_props = dict(color="black", linewidth=1.5)

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    flier_props = dict(
        marker="o",
        markersize=7,
        markerfacecolor="none",
        markeredgecolor=BANDS_COLOR[band],
        markeredgewidth=1.3,
        alpha=0.7,
    )

    box_data, box_labels = [], []
    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]
        if len(disp) < 5:
            continue
        box_data.append(disp)
        box_labels.append(mag_bin_labels[k])

    if len(box_data) == 0:
        ax.set_title(f"band {band} (no data)")
        continue

    bp = ax.boxplot(
        box_data,
        tick_labels=box_labels,
        showfliers=True,
        flierprops=flier_props,
        medianprops=median_props,
        patch_artist=True,
        widths=0.8,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(BANDS_COLOR[band])
        patch.set_alpha(0.5)

    # discret N-points annotation above each box
    for idx, disp in enumerate(box_data, start=1):
        ax.text(
            idx,
            0.97 * ymax_common,
            f"n={len(disp)}",
            ha="center",
            va="top",
            fontsize=10,
            color="darkgray",
            rotation=0,
        )

    n_total = sum(len(d) for d in box_data)
    ax.set_title(f"band {band}  (N={n_total})")
    ax.set_ylim(0, ymax_common)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.2)

for ax in axes[1, :]:
    ax.set_xlabel("magnitude bin")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")


fig.suptitle(
    "Photometric scatter vs. magnitude bin per band -- boxplots "
    f"(outliers = open circles, common y-scale up to {ymax_common:.0f} mmag)",
    y=1.0,
    fontsize=16,
)
plt.tight_layout()

savefig(fig, "mmag_boxplot_vs_magbin_per_band")

plt.show()

### 7.6  Dispersion vs. magnitude bin, violin plots per band, log y-axis (2x3 grid)

Same information as section 7.5 (`2 x 3` grid, one panel per band, x-axis =
magnitude bin), but using **violin plots** instead of boxplots, and with the
dispersion axis **log-binned**: the KDE underlying each violin is computed
on `log10(mmag_meas)` (consistent with the log-spaced binning used in
section 7.2), and the y-axis is displayed with a shared log scale across
all panels. Violins show the full shape of the distribution -- including
any secondary bump or long tail caused by transient atmospheric-
transmission variations -- rather than a five-number summary. Each violin
is colored using the notebook's `BANDS_COLOR` code, and the number of
points per bin is written discreetly above each violin, as in 7.5.

In [ ]:
# ── 7.6  Dispersion vs. magnitude bin, violin plots per band, log y-axis ──
# Same binning as 7.5, but violin plots (full distribution shape) computed
# on log10(mmag_meas), displayed on a shared log-scaled y-axis.

# -- Optional parameters: common y-axis range (mmag) shared by all 6 panels.
# Leave as None to auto-scale from the data, or set explicit values (mmag)
# to fix the range manually, e.g. VIOLIN_YMIN = 2.0, VIOLIN_YMAX = 500.0.
VIOLIN_YMIN = 1
VIOLIN_YMAX = 500

_all_disp = df_all["mmag_meas"].dropna().to_numpy()
_all_disp = _all_disp[np.isfinite(_all_disp) & (_all_disp > 0)]

if VIOLIN_YMIN is None:
    yvmin_common = max(np.nanpercentile(_all_disp, 0.5) * 0.8, _all_disp.min())
else:
    yvmin_common = VIOLIN_YMIN

if VIOLIN_YMAX is None:
    yvmax_common = np.nanpercentile(_all_disp, 99.5) * 1.2
else:
    yvmax_common = VIOLIN_YMAX

log_yvmin, log_yvmax = np.log10(yvmin_common), np.log10(yvmax_common)

# Nice round tick values (mmag) within the shared range
_candidate_ticks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
tick_vals = [v for v in _candidate_ticks if yvmin_common <= v <= yvmax_common]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    box_data, box_labels = [], []
    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]
        if len(disp) < 5:
            continue
        box_data.append(disp)
        box_labels.append(mag_bin_labels[k])

    if len(box_data) == 0:
        ax.set_title(f"band {band} (no data)")
        continue

    log_data = [np.log10(d) for d in box_data]
    positions = np.arange(1, len(log_data) + 1)

    vp = ax.violinplot(
        log_data,
        positions=positions,
        widths=0.8,
        showmedians=True,
        showextrema=True,
    )
    for body in vp["bodies"]:
        body.set_facecolor(BANDS_COLOR[band])
        body.set_edgecolor(BANDS_COLOR[band])
        body.set_alpha(0.5)
    for key in ("cmedians", "cmins", "cmaxes", "cbars"):
        vp[key].set_edgecolor(BANDS_COLOR[band])
        vp[key].set_linewidth(1.2)

    # discreet N-points annotation above each violin
    y_text = log_yvmax - 0.03 * (log_yvmax - log_yvmin)
    for idx, disp in zip(positions, box_data):
        ax.text(
            idx,
            y_text,
            f"n={len(disp)}",
            ha="center",
            va="top",
            fontsize=10,
            color="gray",
            rotation=0,
        )

    n_total = sum(len(d) for d in box_data)
    ax.set_title(f"band {band}  (N={n_total})")
    ax.set_xticks(positions)
    ax.set_xticklabels(box_labels)
    ax.set_ylim(log_yvmin, log_yvmax)
    ax.set_yticks(np.log10(tick_vals))
    ax.set_yticklabels([str(v) for v in tick_vals])
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.2)

for ax in axes[1, :]:
    ax.set_xlabel("magnitude bin")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured -- log scale")

fig.suptitle(
    "Photometric scatter vs. magnitude bin per band -- violin plots, log-binned "
    f"(shared y-range {yvmin_common:.1f}-{yvmax_common:.0f} mmag)",
    y=1.0,
    fontsize=16,
)
plt.tight_layout()

savefig(fig, "mmag_violin_vs_magbin_per_band_logy")

plt.show()